# Misfit Agent — ARC-AGI-3 Tier-1 Submission

**Tier-1 attestation:** this agent uses Spelke core knowledge object priors combined with a hand-authored typed rule grammar over those priors. **No pretrained model weights of any kind.** No language model in the inference path. **Apache-2.0 licensed.**

Source: https://github.com/AtomEons/arc-agi-3-misfit-agent

Mechanically enforced by `tests/test_tier1_attestation.py` — fails CI if any of `transformers`, `openai`, `anthropic`, `llama_cpp`, `huggingface_hub`, `sentence_transformers`, `langchain`, `langgraph`, `smolagents` are imported.


In [ ]:
# Cell 0 — offline wheel install from the bundled competition data.
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
"""
MisfitAgent — Tier-1 Spelke-priors ARC-AGI-3 agent (built notebook bundle).
Generated by scripts/build_notebook.py — do not edit by hand.
Source: https://github.com/AtomEons/arc-agi-3-misfit-agent
"""

# Standard library imports needed across the bundle
from __future__ import annotations

import json
import math
import os
import pathlib
import random
import time
from dataclasses import asdict, dataclass, field
from typing import Any, Optional, Sequence

import numpy as np

# arcengine + framework — supplied by the eval container
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


# ============================================================
# MODULE: config.py
# ============================================================



from dataclasses import dataclass


@dataclass(frozen=True)
class TrackerConfig:
    # Hungarian matching cost weights (Day-4 deliverable).
    # Classification: (a) DERIVED FROM PRIOR — generic geometric distance / topology / color identity.
    # These weights encode the relative importance of position vs shape vs identity for cohesion.
    # NOT tuned on any specific game grid. If we sweep them on public games, we must move them to (c).
    alpha_centroid_dist: float = 1.0
    beta_shape_hamming: float = 0.5
    gamma_color_mismatch: float = 2.0


@dataclass(frozen=True)
class WorldModelConfig:
    # (a) DERIVED FROM PRIOR — same-observation requires same-prediction is the consistency axiom.
    min_observations_for_trust: int = 3
    # (b) BUDGET HEURISTIC — wall-clock per-step simulator target (µs).
    target_us_per_step: int = 50
    # (b) BUDGET HEURISTIC — episodic memory ceiling (states per game).
    max_states_per_game: int = 10_000


@dataclass(frozen=True)
class PseudocountConfig:
    # (a) DERIVED FROM PRIOR — standard novelty-bonus form alpha / sqrt(N+1).
    novelty_alpha: float = 1.0


@dataclass(frozen=True)
class AbstainConfig:
    # (b) BUDGET HEURISTIC — derive from quadratic scoring math.
    # Score = (human/agent)^2. Half-life of marginal score gain is at agent_actions = 2*human_baseline.
    # Default 25 = empirical band where pseudocount novelty typically plateaus on small grids.
    # MUST be re-derived from a published calculation, not asserted — judge auditor flag.
    min_actions_before_abstain: int = 25
    # (a) DERIVED FROM PRIOR — when world-model variance > 20% of predictions, hypothesis class is wrong.
    world_model_variance_threshold: float = 0.20
    # (a) DERIVED FROM PRIOR — pseudocount slope below this = exploration plateau.
    novelty_plateau_slope: float = 0.05


@dataclass(frozen=True)
class MCTSConfig:
    # (b) BUDGET HEURISTIC — PUCT exploration constant; canonical value in AlphaZero literature.
    c_puct: float = 1.41
    # (b) BUDGET HEURISTIC — per-action search depth cap.
    max_depth: int = 6
    # (b) BUDGET HEURISTIC — rollouts per real action; trades quality vs wall clock.
    rollouts_per_action: int = 200
    # (b) BUDGET HEURISTIC — hard timeout per choose_action call (ms).
    hard_timeout_ms: int = 500


@dataclass(frozen=True)
class BudgetConfig:
    # (b) BUDGET HEURISTIC — Kaggle's 9h hard cap minus 5 min cold-start overhead minus 5 min safety.
    wall_clock_kill_seconds: int = 8 * 3600 + 50 * 60   # 8h50m
    # (a) DERIVED FROM PRIOR — the agents.agent.Agent default. Server-side cap may be 80;
    # MUST verify empirically on Day 1 (judge Kaggle-reality must-fix).
    # Raised to 400 in plan but treated as advisory until verified.
    max_actions_per_game: int = 80
    # (b) BUDGET HEURISTIC — anticipated 110 private games per Kaggle eval batch.
    expected_total_games: int = 110


@dataclass(frozen=True)
class FingerprintConfig:
    # (b) BUDGET HEURISTIC — total signature dimensionality.
    total_dim: int = 50


@dataclass(frozen=True)
class FrozenConfig:
    tracker: TrackerConfig = TrackerConfig()
    world_model: WorldModelConfig = WorldModelConfig()
    pseudocount: PseudocountConfig = PseudocountConfig()
    abstain: AbstainConfig = AbstainConfig()
    mcts: MCTSConfig = MCTSConfig()
    budget: BudgetConfig = BudgetConfig()
    fingerprint: FingerprintConfig = FingerprintConfig()


CONFIG = FrozenConfig()

# ============================================================
# MODULE: perceptor.py
# ============================================================



from dataclasses import dataclass, field
from typing import Sequence



Grid = np.ndarray  # 2-D uint8 ARC color grid


@dataclass(frozen=True)
class Obj:
    color: int
    area: int
    bbox: tuple[int, int, int, int]  # (r0, c0, r1, c1) inclusive
    centroid: tuple[float, float]
    touches_edge: bool
    v_symmetric: bool
    h_symmetric: bool


@dataclass
class SceneObservation:
    grid: Grid
    rows: int
    cols: int
    background_color: int
    objects: list[Obj] = field(default_factory=list)
    color_histogram: list[int] = field(default_factory=list)  # length 10
    foreground_cells: int = 0


def _flood_label(grid: Grid, bg: int) -> np.ndarray:
    rows, cols = grid.shape
    labels = np.zeros((rows, cols), dtype=np.int32)
    next_label = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == bg or labels[r, c] != 0:
                continue
            next_label += 1
            stack = [(r, c)]
            target_color = int(grid[r, c])
            while stack:
                rr, cc = stack.pop()
                if rr < 0 or rr >= rows or cc < 0 or cc >= cols:
                    continue
                if labels[rr, cc] != 0:
                    continue
                if int(grid[rr, cc]) != target_color:
                    continue
                labels[rr, cc] = next_label
                stack.extend([(rr+1, cc), (rr-1, cc), (rr, cc+1), (rr, cc-1)])
    return labels


def _is_symmetric_v(sub: Grid) -> bool:
    return bool(np.array_equal(sub, np.flip(sub, axis=1)))


def _is_symmetric_h(sub: Grid) -> bool:
    return bool(np.array_equal(sub, np.flip(sub, axis=0)))


def _background_color(grid: Grid) -> int:
    if (grid == 0).any():
        return 0
    counts = np.bincount(grid.ravel(), minlength=10)
    return int(np.argmax(counts))


def perceive_grid(grid: Grid) -> SceneObservation:
    grid = np.asarray(grid, dtype=np.int32)
    if grid.ndim != 2:
        raise ValueError(f"perceive_grid expects 2-D grid, got shape {grid.shape}")
    rows, cols = grid.shape
    bg = _background_color(grid)
    labels = _flood_label(grid, bg)
    n_labels = int(labels.max())

    objects: list[Obj] = []
    for lab in range(1, n_labels + 1):
        mask = labels == lab
        ys, xs = np.where(mask)
        if ys.size == 0:
            continue
        r0, r1 = int(ys.min()), int(ys.max())
        c0, c1 = int(xs.min()), int(xs.max())
        sub = grid[r0:r1+1, c0:c1+1]
        sub_mask = mask[r0:r1+1, c0:c1+1].astype(np.int32) * (sub != bg).astype(np.int32)
        color = int(grid[ys[0], xs[0]])
        touches = bool(r0 == 0 or c0 == 0 or r1 == rows-1 or c1 == cols-1)
        v_sym = _is_symmetric_v(sub_mask)
        h_sym = _is_symmetric_h(sub_mask)
        centroid = (float(ys.mean()), float(xs.mean()))
        objects.append(Obj(
            color=color,
            area=int(ys.size),
            bbox=(r0, c0, r1, c1),
            centroid=centroid,
            touches_edge=touches,
            v_symmetric=v_sym,
            h_symmetric=h_sym,
        ))

    # Sort by area descending (largest first)
    objects.sort(key=lambda o: -o.area)

    hist = np.bincount(grid.ravel(), minlength=10)[:10].tolist()
    fg = int((grid != bg).sum())

    return SceneObservation(
        grid=grid,
        rows=rows,
        cols=cols,
        background_color=bg,
        objects=objects,
        color_histogram=list(map(int, hist)),
        foreground_cells=fg,
    )


def perceive_frame(frame: Sequence) -> SceneObservation:
    arr = np.asarray(frame, dtype=np.int32)
    if arr.ndim == 3:
        # Multi-plane: use the first plane. Higher-order spatial reasoning over
        # multiple planes is a Day-N+ extension under the geometry prior.
        arr = arr[0]
    if arr.ndim != 2:
        raise ValueError(f"perceive_frame expects 2-D or 3-D grid, got shape {arr.shape}")
    return perceive_grid(arr)


def grid_diff(a: Grid, b: Grid) -> tuple[int, int, int]:
    a = np.asarray(a, dtype=np.int32)
    b = np.asarray(b, dtype=np.int32)
    if a.shape != b.shape:
        return (max(a.size, b.size), 0, 0)
    bg_a = _background_color(a)
    bg_b = _background_color(b)
    a_bg = a == bg_a
    b_bg = b == bg_b
    bg_to_fg = int(((a_bg) & (~b_bg)).sum())
    fg_to_bg = int(((~a_bg) & (b_bg)).sum())
    changed = int((a != b).sum())
    return (changed, bg_to_fg, fg_to_bg)

# ============================================================
# MODULE: episode.py
# ============================================================



from typing import Any, Optional



@dataclass
class ActionRecord:
    action_name: str         # e.g. "ACTION1", "ACTION6"
    action_value: int        # numeric enum value
    data: dict               # {"x": int, "y": int} for complex actions
    pre_levels_completed: int
    post_levels_completed: Optional[int] = None
    cells_changed: Optional[int] = None
    triggered_win: bool = False


@dataclass
class EpisodeTracker:
    game_id: str
    scenes: list[SceneObservation] = field(default_factory=list)
    action_history: list[ActionRecord] = field(default_factory=list)
    last_action_rationale: str = ""

    # Lightweight induced rule beliefs (Day 2+ will fill these from transitions).
    # All keys are *observed* dynamics, not hand-crafted task families.
    transition_signals: dict[str, Any] = field(default_factory=dict)

    def observe(self, latest_frame: Any, scene: SceneObservation) -> None:
        prior_scene = self.scenes[-1] if self.scenes else None
        self.scenes.append(scene)

        # If we have a prior action record awaiting a follow-up, close it.
        if self.action_history and self.action_history[-1].post_levels_completed is None:
            rec = self.action_history[-1]
            rec.post_levels_completed = int(latest_frame.levels_completed)
            if prior_scene is not None and scene.grid.shape == prior_scene.grid.shape:
                changed, _, _ = grid_diff(prior_scene.grid, scene.grid)
                rec.cells_changed = changed
            rec.triggered_win = bool(getattr(latest_frame, "state", None) and
                                     str(latest_frame.state).endswith("WIN"))
            self._update_transition_signals(rec, prior_scene, scene)

    def observe_terminal(self, latest_frame: Any) -> None:
        if self.action_history and self.action_history[-1].post_levels_completed is None:
            self.action_history[-1].post_levels_completed = int(latest_frame.levels_completed)

    def record_action(self, action_name: str, action_value: int, data: dict,
                      pre_levels_completed: int) -> None:
        self.action_history.append(ActionRecord(
            action_name=action_name,
            action_value=action_value,
            data=dict(data) if data else {},
            pre_levels_completed=pre_levels_completed,
        ))

    def set_rationale(self, rationale: str) -> None:
        self.last_action_rationale = rationale

    def winning_actions(self) -> list[ActionRecord]:
        return [a for a in self.action_history if a.triggered_win]

    def level_advancing_actions(self) -> list[ActionRecord]:
        out = []
        for a in self.action_history:
            if a.post_levels_completed is not None and a.post_levels_completed > a.pre_levels_completed:
                out.append(a)
        return out

    def _update_transition_signals(self, rec: ActionRecord,
                                    prior: SceneObservation,
                                    after: SceneObservation) -> None:
        if prior is None:
            return
        action_key = rec.action_name
        bucket = self.transition_signals.setdefault(action_key, {
            "total": 0,
            "cells_changed_sum": 0,
            "level_advances": 0,
            "object_count_delta_sum": 0,
        })
        bucket["total"] += 1
        bucket["cells_changed_sum"] += rec.cells_changed or 0
        if rec.post_levels_completed is not None and rec.post_levels_completed > rec.pre_levels_completed:
            bucket["level_advances"] += 1
        bucket["object_count_delta_sum"] += len(after.objects) - len(prior.objects)

# ============================================================
# MODULE: fingerprint.py
# ============================================================







FINGERPRINT_DIM = 50


def fingerprint_episode(tracker: EpisodeTracker) -> np.ndarray:
    v = np.zeros(FINGERPRINT_DIM, dtype=np.float32)
    scenes = tracker.scenes
    if not scenes:
        return v

    rows = [s.rows for s in scenes]
    cols = [s.cols for s in scenes]
    v[0] = float(np.mean(rows)) / max(float(np.mean(cols)), 1.0)

    same_shape = sum(1 for i in range(1, len(scenes))
                     if scenes[i].grid.shape == scenes[i-1].grid.shape)
    v[1] = same_shape / max(len(scenes) - 1, 1)

    obj_counts = [len(s.objects) for s in scenes]
    v[2] = float(np.mean(obj_counts)) if obj_counts else 0.0
    v[3] = float(obj_counts[-1] - obj_counts[0]) if len(obj_counts) > 1 else 0.0

    def _mean_safe(arr): return float(np.mean(arr)) if arr else 0.0
    v[4] = _mean_safe([sum(o.v_symmetric for o in s.objects) / max(len(s.objects), 1)
                       for s in scenes])
    v[5] = _mean_safe([sum(o.h_symmetric for o in s.objects) / max(len(s.objects), 1)
                       for s in scenes])
    v[6] = _mean_safe([sum(o.touches_edge for o in s.objects) / max(len(s.objects), 1)
                       for s in scenes])
    v[7] = _mean_safe([(s.objects[0].area / max(s.rows*s.cols, 1)) if s.objects else 0.0
                       for s in scenes])
    v[8] = _mean_safe([s.foreground_cells / max(s.rows*s.cols, 1) for s in scenes])
    v[9] = float(math.log(1 + len(scenes)))

    # Palette density per color 0..9 (mean over scenes)
    palette_means = np.zeros(10, dtype=np.float32)
    for s in scenes:
        denom = max(s.rows * s.cols, 1)
        for c in range(10):
            palette_means[c] += s.color_histogram[c] / denom
    palette_means /= max(len(scenes), 1)
    v[10:20] = palette_means

    # Palette delta — last vs first scene
    first_p = np.array(scenes[0].color_histogram, dtype=np.float32) / max(
        scenes[0].rows * scenes[0].cols, 1)
    last_p = np.array(scenes[-1].color_histogram, dtype=np.float32) / max(
        scenes[-1].rows * scenes[-1].cols, 1)
    v[20:30] = (last_p - first_p)

    # Action-effect signature & level-advance rate per action enum slot.
    action_slots = ["RESET", "ACTION1", "ACTION2", "ACTION3", "ACTION4",
                    "ACTION5", "ACTION6", "ACTION7"]
    for i, name in enumerate(action_slots):
        bucket = tracker.transition_signals.get(name)
        if not bucket or bucket["total"] == 0:
            continue
        v[30 + i] = bucket["cells_changed_sum"] / bucket["total"]
        v[38 + i] = bucket["level_advances"] / bucket["total"]

    return v


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# ============================================================
# MODULE: resonance.py
# ============================================================



from typing import Optional




def default_library_path() -> pathlib.Path:
    if os.name == "nt":
        base = pathlib.Path(os.environ.get("LOCALAPPDATA", str(pathlib.Path.home())))
    else:
        base = pathlib.Path.home() / ".local" / "share"
    return base / "misfit-agent" / "resonance_library.jsonl"


@dataclass
class LibraryEntry:
    game_id: str
    fingerprint: list[float]
    winning_policy: list[dict]  # serialized ActionRecord list
    composite_score: float
    solved_at_unix: float
    source: str = "self-solved"


@dataclass
class ResonanceLibrary:
    path: pathlib.Path
    entries: list[LibraryEntry] = field(default_factory=list)
    _pending: list[LibraryEntry] = field(default_factory=list)

    @classmethod
    def load_or_create(cls, path: str | pathlib.Path) -> "ResonanceLibrary":
        path = pathlib.Path(path)
        lib = cls(path=path)
        if not path.exists():
            return lib
        try:
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        d = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    if d.get("source") != "self-solved":
                        # Tier-1 honesty: silently skip non-self-solved entries.
                        continue
                    lib.entries.append(LibraryEntry(**d))
        except OSError:
            pass
        return lib

    def find_k_nearest(self, query: np.ndarray, k: int = 5
                       ) -> list[tuple[LibraryEntry, float]]:
        if not self.entries:
            return []
        scored = []
        for e in self.entries:
            sim = cosine(query, np.asarray(e.fingerprint, dtype=np.float32))
            scored.append((e, sim))
        scored.sort(key=lambda x: -x[1])
        return scored[:k]

    def retrieve_policy_seeds(self, query: np.ndarray, k: int = 5
                              ) -> list[list[dict]]:
        seen: set[str] = set()
        out: list[list[dict]] = []
        for e, _ in self.find_k_nearest(query, k):
            sig = json.dumps([a["action_value"] for a in e.winning_policy])
            if sig in seen:
                continue
            seen.add(sig)
            out.append(e.winning_policy)
        return out

    def record_solved(self, fingerprint: np.ndarray, winning_policy: list[ActionRecord],
                       composite_score: float, source: str = "self-solved",
                       game_id: Optional[str] = None) -> LibraryEntry:
        if source != "self-solved":
            raise ValueError(
                f"Tier-1 honesty: library only accepts self-solved entries, got {source!r}"
            )
        gid = game_id or (winning_policy[0].action_name if winning_policy else "unknown")
        entry = LibraryEntry(
            game_id=gid,
            fingerprint=[float(x) for x in fingerprint.tolist()],
            winning_policy=[asdict(a) for a in winning_policy],
            composite_score=float(composite_score),
            solved_at_unix=time.time(),
            source=source,
        )
        self.entries.append(entry)
        self._pending.append(entry)
        return entry

    def flush_to_disk(self) -> int:
        if not self._pending:
            return 0
        self.path.parent.mkdir(parents=True, exist_ok=True)
        n = 0
        with self.path.open("a", encoding="utf-8") as f:
            for e in self._pending:
                f.write(json.dumps(asdict(e)) + "\n")
                n += 1
        self._pending.clear()
        return n

# ============================================================
# MODULE: rules/no_op.py
# ============================================================






@dataclass
class NoOp:

    object_class: int   # which color/class this rule applies to
    fitted: bool = False

    def fit(self, observations: list[dict]) -> bool:
        for obs in observations:
            pre = obs.get("pre_objects_of_class", [])
            post = obs.get("post_objects_of_class", [])
            if len(pre) != len(post):
                return False
            for p, q in zip(pre, post):
                if p["centroid"] != q["centroid"]:
                    return False
                if p["area"] != q["area"]:
                    return False
        self.fitted = True
        return True

    def predict(self, grid: np.ndarray, action_name: str) -> np.ndarray:
        return grid.copy()

# ============================================================
# MODULE: rules/translate.py
# ============================================================






@dataclass
class Translate:
    object_class: int
    dx_per_action: dict[str, int] = field(default_factory=dict)
    dy_per_action: dict[str, int] = field(default_factory=dict)
    fitted: bool = False
    consistency_score: float = 0.0

    def fit(self, observations: list[dict]) -> bool:
        per_action_deltas: dict[str, list[tuple[float, float]]] = {}
        for obs in observations:
            action = obs.get("action_name", "")
            pre = obs.get("pre_objects_of_class", [])
            post = obs.get("post_objects_of_class", [])
            if len(pre) != 1 or len(post) != 1:
                continue  # Translate is single-object for now
            pre_r, pre_c = pre[0]["centroid"]
            post_r, post_c = post[0]["centroid"]
            per_action_deltas.setdefault(action, []).append(
                (post_r - pre_r, post_c - pre_c)
            )

        if not per_action_deltas:
            return False

        total = 0
        consistent = 0
        for action, deltas in per_action_deltas.items():
            if not deltas:
                continue
            dy = round(float(np.median([d[0] for d in deltas])))
            dx = round(float(np.median([d[1] for d in deltas])))
            for d in deltas:
                total += 1
                if abs(d[0] - dy) < 0.5 and abs(d[1] - dx) < 0.5:
                    consistent += 1
            self.dy_per_action[action] = dy
            self.dx_per_action[action] = dx

        if total == 0:
            return False

        self.consistency_score = consistent / total
        self.fitted = self.consistency_score >= 0.8   # 80% consistency threshold
        return self.fitted

    def predict(self, grid: np.ndarray, action_name: str) -> np.ndarray:
        dx = self.dx_per_action.get(action_name, 0)
        dy = self.dy_per_action.get(action_name, 0)
        if dx == 0 and dy == 0:
            return grid.copy()
        out = grid.copy()
        rows, cols = out.shape
        mask = out == self.object_class
        if not mask.any():
            return out
        ys, xs = np.where(mask)
        # Clear current positions
        out[mask] = _background_color(grid)
        # Place at translated positions, clipped to grid bounds
        new_ys = np.clip(ys + dy, 0, rows - 1)
        new_xs = np.clip(xs + dx, 0, cols - 1)
        out[new_ys, new_xs] = self.object_class
        return out


def _background_color(grid: np.ndarray) -> int:
    if (grid == 0).any():
        return 0
    counts = np.bincount(grid.ravel(), minlength=10)
    return int(np.argmax(counts))

# ============================================================
# MODULE: world_model.py
# ============================================================







RuleInstance = Translate | NoOp


@dataclass
class WorldModel:

    rules: list[RuleInstance] = field(default_factory=list)
    observations_per_action: dict[str, int] = field(default_factory=dict)
    confirmed_transitions: dict[str, int] = field(default_factory=dict)

    def fit(self, observations: list[dict]) -> dict[str, float]:
        # Group observations by object_class so each rule fits within-class
        by_class: dict[int, list[dict]] = {}
        for obs in observations:
            for c in obs.get("classes_involved", []):
                by_class.setdefault(c, []).append(obs)

        scores: dict[str, float] = {}
        new_rules: list[RuleInstance] = []

        for cls, cls_obs in by_class.items():
            tr = Translate(object_class=cls)
            if tr.fit(cls_obs):
                new_rules.append(tr)
                scores[f"Translate(class={cls})"] = tr.consistency_score
                continue
            no = NoOp(object_class=cls)
            if no.fit(cls_obs):
                new_rules.append(no)
                scores[f"NoOp(class={cls})"] = 1.0

        # Update transition-count bookkeeping
        for obs in observations:
            action = obs.get("action_name", "")
            self.observations_per_action[action] = (
                self.observations_per_action.get(action, 0) + 1
            )

        self.rules = new_rules
        return scores

    def predict(
        self,
        grid: np.ndarray,
        action_name: str,
    ) -> tuple[np.ndarray, float]:
        if action_name == "RESET":
            # Can't predict reset effects without level-start observations;
            # caller should treat as low-confidence.
            return grid.copy(), 0.0

        obs_count = self.observations_per_action.get(action_name, 0)
        threshold = CONFIG.world_model.min_observations_for_trust
        if obs_count < threshold or not self.rules:
            return grid.copy(), 0.0

        predicted = grid.copy()
        rules_fired = 0
        for rule in self.rules:
            try:
                next_grid = rule.predict(predicted, action_name)
            except Exception:
                continue
            if not np.array_equal(next_grid, predicted):
                predicted = next_grid
                rules_fired += 1

        if rules_fired == 0:
            return predicted, 0.5  # rules exist but none fire — partial confidence

        return predicted, 1.0

    def coverage(self) -> float:
        if not self.observations_per_action:
            return 0.0
        threshold = CONFIG.world_model.min_observations_for_trust
        trusted = sum(1 for n in self.observations_per_action.values() if n >= threshold)
        return trusted / len(self.observations_per_action)

    def has_class_coverage(self, object_class: int) -> bool:
        return any(getattr(r, "object_class", None) == object_class for r in self.rules)

# ============================================================
# MODULE: click_quantizer.py
# ============================================================






GRID_MAX_X = 63
GRID_MAX_Y = 63


@dataclass(frozen=True)
class ClickCandidate:
    x: int
    y: int
    rationale: str    # priors-only rationale string (no game-specific text)
    source: str       # "centroid" | "bbox_corner" | "edge_midpoint" | "quadrant_fallback"


def _clip(v: int, lo: int = 0, hi: int = GRID_MAX_X) -> int:
    return max(lo, min(hi, v))


def click_candidates(scene: SceneObservation, max_candidates: int = 20
                     ) -> list[ClickCandidate]:
    seen: set[tuple[int, int]] = set()
    out: list[ClickCandidate] = []

    # 1. Centroids of all detected objects (already area-sorted descending).
    for i, obj in enumerate(scene.objects):
        cr, cc = obj.centroid
        x, y = _clip(int(round(cc))), _clip(int(round(cr)))
        key = (x, y)
        if key in seen:
            continue
        seen.add(key)
        out.append(ClickCandidate(
            x=x, y=y,
            rationale=f"centroid of object rank {i} (color={obj.color}, area={obj.area})",
            source="centroid",
        ))
        if len(out) >= max_candidates:
            return out

    # 2. BBox corners of the top-3 objects — captures grab handles and
    #    "destination" cells common in object-puzzle games.
    for i, obj in enumerate(scene.objects[:3]):
        r0, c0, r1, c1 = obj.bbox
        for (yy, xx, corner) in [(r0, c0, "TL"), (r0, c1, "TR"),
                                  (r1, c0, "BL"), (r1, c1, "BR")]:
            x, y = _clip(int(xx)), _clip(int(yy))
            key = (x, y)
            if key in seen:
                continue
            seen.add(key)
            out.append(ClickCandidate(
                x=x, y=y,
                rationale=f"bbox {corner} of object rank {i}",
                source="bbox_corner",
            ))
            if len(out) >= max_candidates:
                return out

    # 3. Edge midpoints — captures "edge target" patterns the perceptor
    #    flagged via touches_edge but the centroid missed.
    for i, obj in enumerate(scene.objects[:3]):
        if not obj.touches_edge:
            continue
        r0, c0, r1, c1 = obj.bbox
        for (yy, xx, side) in [
            (r0, (c0 + c1) // 2, "top"),
            (r1, (c0 + c1) // 2, "bottom"),
            ((r0 + r1) // 2, c0, "left"),
            ((r0 + r1) // 2, c1, "right"),
        ]:
            x, y = _clip(int(xx)), _clip(int(yy))
            key = (x, y)
            if key in seen:
                continue
            seen.add(key)
            out.append(ClickCandidate(
                x=x, y=y,
                rationale=f"edge midpoint ({side}) of edge-touching object rank {i}",
                source="edge_midpoint",
            ))
            if len(out) >= max_candidates:
                return out

    # 4. 9-quadrant geometric fallback — generic coverage when objectness
    #    is empty (e.g. blank-grid start state). Pure geometry prior.
    if scene.rows > 0 and scene.cols > 0:
        max_r = scene.rows - 1
        max_c = scene.cols - 1
        grid_pts = [
            (max_r * a // 2, max_c * b // 2)
            for a in (0, 1, 2)
            for b in (0, 1, 2)
        ]
        for (yy, xx) in grid_pts:
            x, y = _clip(int(xx)), _clip(int(yy))
            key = (x, y)
            if key in seen:
                continue
            seen.add(key)
            out.append(ClickCandidate(
                x=x, y=y,
                rationale="9-quadrant geometric fallback",
                source="quadrant_fallback",
            ))
            if len(out) >= max_candidates:
                return out

    return out


def best_click_candidate(scene: SceneObservation,
                         policy_seeds_xy: Optional[list[tuple[int, int]]] = None
                         ) -> ClickCandidate:
    candidates = click_candidates(scene)
    if not candidates:
        # Truly empty — center the grid.
        cx = (scene.cols - 1) // 2 if scene.cols else 0
        cy = (scene.rows - 1) // 2 if scene.rows else 0
        return ClickCandidate(
            x=_clip(cx), y=_clip(cy),
            rationale="empty scene; center-of-grid fallback",
            source="empty_fallback",
        )

    if policy_seeds_xy:
        # Pick the candidate closest to ANY seed coordinate.
        best = candidates[0]
        best_dist = float("inf")
        for c in candidates:
            for sx, sy in policy_seeds_xy:
                d = (c.x - sx) ** 2 + (c.y - sy) ** 2
                if d < best_dist:
                    best_dist = d
                    best = c
        return best

    return candidates[0]

# ============================================================
# MODULE: action_search.py
# ============================================================





from arcengine import GameAction  # type: ignore[import-not-found]



def _name_of(action: Any) -> str:
    return getattr(action, "name", str(action))


def _action_dud_score(tracker: EpisodeTracker, name: str) -> float:
    bucket = tracker.transition_signals.get(name)
    if not bucket or bucket["total"] == 0:
        return 0.0
    if bucket["cells_changed_sum"] == 0 and bucket["level_advances"] == 0:
        return 1.0
    return 0.0


def _action_advance_rate(tracker: EpisodeTracker, name: str) -> float:
    bucket = tracker.transition_signals.get(name)
    if not bucket or bucket["total"] == 0:
        return 0.0
    return bucket["level_advances"] / bucket["total"]


def _world_model_lookahead_score(
    world_model: Optional[WorldModel],
    grid: np.ndarray,
    action_name: str,
    tracker: EpisodeTracker,
) -> tuple[float, float]:
    if world_model is None:
        return 0.0, 0.0
    try:
        predicted, conf = world_model.predict(grid, action_name)
    except Exception:
        return 0.0, 0.0
    if conf <= 0.0:
        return 0.0, 0.0
    bonus = 0.0
    if not np.array_equal(predicted, grid):
        bonus += 0.5
        bucket = tracker.transition_signals.get(action_name)
        if not bucket or bucket["level_advances"] == 0:
            bonus += 0.3
        elif bucket["cells_changed_sum"] > 0:
            bonus += 0.2
    return bonus, conf


def select_action(
    scene: SceneObservation,
    tracker: EpisodeTracker,
    available_actions: Sequence[Any],
    policy_seeds: Sequence[Sequence[dict]],
    action_budget_remaining: int,
    world_model: Optional[WorldModel] = None,
) -> GameAction:

    if not available_actions:
        tracker.record_action("RESET", int(GameAction.RESET.value), {},
                              pre_levels_completed=0)
        tracker.set_rationale("no available_actions reported; resetting")
        return GameAction.RESET

    # Step index → resonance seed votes.
    step_idx = len(tracker.action_history)
    seed_votes: dict[str, int] = {}
    seed_xy_hints: list[tuple[int, int]] = []
    for policy in policy_seeds:
        if step_idx < len(policy):
            entry = policy[step_idx]
            n = entry.get("action_name", "")
            if n:
                seed_votes[n] = seed_votes.get(n, 0) + 1
            data = entry.get("data") or {}
            if "x" in data and "y" in data:
                seed_xy_hints.append((int(data["x"]), int(data["y"])))

    # Score each available action.
    scored: list[tuple[float, Any, float]] = []  # (weight, action, wm_conf)
    for action in available_actions:
        name = _name_of(action)
        if _action_dud_score(tracker, name) >= 1.0 and action_budget_remaining > 5:
            continue
        weight = 1.0
        weight += 2.0 * _action_advance_rate(tracker, name)
        weight += 1.5 * seed_votes.get(name, 0)
        wm_bonus, wm_conf = _world_model_lookahead_score(
            world_model, scene.grid, name, tracker
        )
        weight += wm_bonus
        scored.append((weight, action, wm_conf))

    if not scored:
        action = random.choice(list(available_actions))
        chosen_name = _name_of(action)
        rationale = "all-actions-known-dud; random tiebreak"
        wm_conf_chosen = 0.0
    else:
        max_w = max(w for w, _, _ in scored)
        top = [(a, c) for w, a, c in scored if w == max_w]
        action, wm_conf_chosen = random.choice(top)
        chosen_name = _name_of(action)
        rationale = (
            f"priors+wm pick: advance={_action_advance_rate(tracker, chosen_name):.2f}, "
            f"seed_votes={seed_votes.get(chosen_name, 0)}, "
            f"wm_conf={wm_conf_chosen:.2f}"
        )

    # ACTION6 — quantize click coordinate via objectness prior, biased by
    # seed (x,y) hints if any.
    data: dict = {}
    if hasattr(action, "is_complex") and action.is_complex():
        cand = best_click_candidate(scene, policy_seeds_xy=seed_xy_hints or None)
        data = {"x": cand.x, "y": cand.y}
        action.set_data(data)
        rationale += f"; click=({cand.x},{cand.y}) {cand.source} [{cand.rationale}]"

    pre_lv = (
        tracker.action_history[-1].post_levels_completed
        if tracker.action_history
        and tracker.action_history[-1].post_levels_completed is not None
        else 0
    )
    tracker.record_action(chosen_name, int(getattr(action, "value", 0)), data,
                          pre_levels_completed=pre_lv)
    tracker.set_rationale(rationale)
    return action

# ============================================================
# MODULE: misfit_agent.py
# ============================================================








class Misfit(Agent):

    # StochasticGoose-confirmed: server does NOT enforce per-game cap.
    # 8h55m wall-clock self-kill is the real budget owner.
    MAX_ACTIONS = float("inf")  # type: ignore[assignment]
    LIBRARY_PATH: Optional[str] = None  # None → use default_library_path()
    HARD_WALL_CLOCK_SECONDS = 8 * 3600 + 50 * 60   # 8h50m — judges' Kaggle-reality fix

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.tracker = EpisodeTracker(game_id=self.game_id)
        path = self.LIBRARY_PATH or str(default_library_path())
        self.library = ResonanceLibrary.load_or_create(path)
        self.world_model = WorldModel()
        self._start_time = time.time()
        self._wm_observations: list[dict] = []

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MAX_ACTIONS}"

    def _has_time_elapsed(self) -> bool:
        return (time.time() - self._start_time) >= self.HARD_WALL_CLOCK_SECONDS

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        try:
            if latest_frame.state is GameState.WIN:
                return True
            if self._has_time_elapsed():
                return True
        except Exception:
            return True  # bail out on unexpected state — never block the framework
        return False

    def choose_action(
        self,
        frames: list[FrameData],
        latest_frame: FrameData,
    ) -> GameAction:
        try:
            # Game not started or game-over — reset.
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                self.tracker.observe_terminal(latest_frame)
                action = GameAction.RESET
                action.reasoning = "game needs reset"
                return action

            # Perceive the current frame under Spelke priors.
            scene = perceive_frame(latest_frame.frame)
            self.tracker.observe(latest_frame, scene)

            # Refit the world model whenever a fresh transition just landed.
            self._maybe_refit_world_model()

            # Pull resonance seeds — prior winning policies for similar episodes.
            signature = fingerprint_episode(self.tracker)
            seeds = self.library.retrieve_policy_seeds(signature, k=5)

            # Resolve available actions. Per StochasticGoose: the gateway sends
            # raw ints [1,2,...,6,7] rather than GameAction enum members.
            raw_avail = getattr(latest_frame, "available_actions", None) or []
            available = []
            for a in raw_avail:
                aid = a.value if hasattr(a, "value") else int(a)
                try:
                    available.append(GameAction.from_id(aid))
                except Exception:
                    # If from_id is missing, try enum lookup by value.
                    for ga in GameAction:
                        if int(getattr(ga, "value", -1)) == aid:
                            available.append(ga)
                            break

            # Select via priors + world-model lookahead + click quantization.
            action = select_action(
                scene=scene,
                tracker=self.tracker,
                available_actions=available,
                policy_seeds=seeds,
                action_budget_remaining=10_000,  # wall-clock bound, not action bound
                world_model=self.world_model,
            )

            # Attach the rationale the substrate produced (for traceability).
            rationale = self.tracker.last_action_rationale or "priors-only fallback"
            if action.is_simple():
                action.reasoning = rationale
            elif action.is_complex():
                action.reasoning = {
                    "desired_action": str(getattr(action, "value", "")),
                    "rationale": rationale,
                }
            return action
        except Exception as e:
            # Never crash the framework — defensive fallback.
            try:
                action = GameAction.ACTION1
                action.reasoning = f"fallback after exception: {type(e).__name__}: {e}"
                return action
            except Exception:
                return GameAction.RESET

    def _maybe_refit_world_model(self) -> None:
        if len(self.tracker.scenes) < 2 or not self.tracker.action_history:
            return
        last_action = self.tracker.action_history[-1]
        prev_scene = self.tracker.scenes[-2]
        curr_scene = self.tracker.scenes[-1]
        if prev_scene.grid.shape != curr_scene.grid.shape:
            return

        classes_involved = sorted(set(int(o.color) for o in prev_scene.objects)
                                  | set(int(o.color) for o in curr_scene.objects))
        # Group object summaries by color for the rule.fit consumers.
        def _grouped(scene):
            by_color: dict[int, list[dict]] = {}
            for o in scene.objects:
                by_color.setdefault(int(o.color), []).append({
                    "centroid": o.centroid,
                    "area": int(o.area),
                })
            return by_color

        prev_by_color = _grouped(prev_scene)
        curr_by_color = _grouped(curr_scene)

        # Emit one observation per class so the composer can fit per-class rules.
        for cls in classes_involved:
            obs = {
                "action_name": last_action.action_name,
                "classes_involved": [cls],
                "pre_objects_of_class": prev_by_color.get(cls, []),
                "post_objects_of_class": curr_by_color.get(cls, []),
            }
            self._wm_observations.append(obs)

        # Refit every 5 steps to bound cost.
        if len(self.tracker.action_history) % 5 == 0:
            try:
                self.world_model.fit(self._wm_observations)
            except Exception:
                pass

    def cleanup(self, scorecard: Optional[Any] = None) -> None:
        if self.frames and self.frames[-1].state is GameState.WIN:
            self.library.record_solved(
                fingerprint=fingerprint_episode(self.tracker),
                winning_policy=self.tracker.action_history.copy(),
                composite_score=self._compute_composite_score(),
                source="self-solved",
            )
            self.library.flush_to_disk()
        super().cleanup(scorecard)

    def _compute_composite_score(self) -> float:
        if not self.action_counter:
            return 1.0
        budget_left = max(0, self.MAX_ACTIONS - self.action_counter)
        return budget_left / self.MAX_ACTIONS

# Framework expects `MyAgent` to subclass Agent.
MyAgent = Misfit


In [ ]:
# Cell 2 — KAGGLE_IS_COMPETITION_RERUN bootstrap (Phase B execution).
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the eval gateway to come up.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework to a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Install our agent module into agents/templates/.
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Slim __init__.py — skip langgraph/smolagents/openai whose deps
    # are NOT in the offline wheel set. Eagerly importing them crashes
    # the whole framework load before MyAgent gets registered.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
''')

    # .env — gateway endpoint per the rerun contract.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # Run.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
# Cell 3 — non-rerun dummy submission for Phase A validation.
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Wrote dummy submission.parquet for Phase A.')
